In [1]:
import pandas as pd
import numpy as np
import re

df_sqli = pd.read_csv('datasets/sqli_clean.csv')
df_xss  = pd.read_csv('datasets/xss_clean.csv')

print("=== LOADED CLEANED DATASETS ===")
print(f"SQLi : {df_sqli.shape}  labels={df_sqli['label'].value_counts().to_dict()}")
print(f"XSS  : {df_xss.shape}   labels={df_xss['label'].value_counts().to_dict()}")
print()
print("Sample SQLi attack payload:")
print(" ", df_sqli[df_sqli['label']==1]['payload'].iloc[0][:80])
print()
print("Sample XSS attack payload:")
print(" ", df_xss[df_xss['label']==1]['payload'].iloc[0][:80])
print()
print("Sample XSS benign (Wikipedia HTML — why length is not a good feature):")
print(" ", df_xss[df_xss['label']==0]['payload'].iloc[0][:80])

=== LOADED CLEANED DATASETS ===
SQLi : (30906, 3)  labels={0: 19529, 1: 11377}
XSS  : (10914, 3)   labels={1: 7323, 0: 3591}

Sample SQLi attack payload:
  " or pg_sleep  (  __TIME__  )  --

Sample XSS attack payload:
  <tt onmouseover="alert(1)">test</tt>

Sample XSS benign (Wikipedia HTML — why length is not a good feature):
  <li><a href="/wiki/File:Socrates.png" class="image"><img alt="Socrates.png" src=


In [2]:
SPECIAL_CHARS = [
    "'", '"', '<', '>', '--', ';', '(', ')', '=',
    '%27', '%3C', '%3E', '%22', '%28', '%29',
    'OR ', 'AND ', 'NULL'
]

KEYWORDS = [
    # SQL injection keywords
    'select', 'union', 'drop', 'insert', 'update', 'delete',
    'where', 'from', 'having', 'exec', 'execute',
    'sleep', 'benchmark', 'waitfor', 'delay',
    'information_schema', 'xp_', 'sp_',
    'cast', 'convert', 'or 1=1', 'and 1=1',
    # XSS keywords
    'script', 'alert', 'onerror', 'onload', 'onmouseover',
    'javascript', 'eval', 'document', 'window',
    'iframe', 'onblur', 'onfocus', 'ondrag', 'onpointer',
    'autofocus', 'tabindex', 'ondeactivate',
    # Path traversal
    '../', 'etc/passwd'
]

def extract_features(text):
    text_str   = str(text)
    text_lower = text_str.lower()
    text_upper = text_str.upper()
    pl = len(text_str)
    sc = sum(text_upper.count(c.upper()) for c in SPECIAL_CHARS)
    kw = sum(1 for kw in KEYWORDS if kw.lower() in text_lower)
    return pl, sc, kw

def build_features(df):
    feats = df['payload'].apply(extract_features)
    df = df.copy()
    df['payload_length']     = [f[0] for f in feats]
    df['special_char_count'] = [f[1] for f in feats]
    df['keyword_count']      = [f[2] for f in feats]
    return df

df_sqli = build_features(df_sqli)
df_xss  = build_features(df_xss)

FEATURES = ['payload_length', 'special_char_count', 'keyword_count']

print("=== FEATURE AVERAGES BY LABEL ===")
print("\nSQLi dataset:")
print(df_sqli.groupby('label')[FEATURES].mean().round(2))
print("\nXSS dataset:")
print(df_xss.groupby('label')[FEATURES].mean().round(2))

=== FEATURE AVERAGES BY LABEL ===

SQLi dataset:
       payload_length  special_char_count  keyword_count
label                                                   
0               39.93                1.67           1.40
1              118.66               12.17           1.79

XSS dataset:
       payload_length  special_char_count  keyword_count
label                                                   
0              250.31               27.71           0.09
1               62.60               10.50           1.74


In [3]:
from sklearn.model_selection import train_test_split

df_combined = pd.concat([df_sqli, df_xss], ignore_index=True)
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

X = df_combined[FEATURES]
y = df_combined['label']

print("=== COMBINED DATASET ===")
print(f"Total rows : {len(df_combined):,}")
print(f"Benign (0) : {(y==0).sum():,}")
print(f"Attack (1) : {(y==1).sum():,}")
print()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# stratify=y ensures same 0/1 ratio in train and test sets

print(f"Train rows : {len(X_train):,}")
print(f"Test  rows : {len(X_test):,}")

=== COMBINED DATASET ===
Total rows : 41,820
Benign (0) : 23,120
Attack (1) : 18,700

Train rows : 33,456
Test  rows : 8,364


In [5]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
model_lr.fit(X_train, y_train)

print("LogisticRegression trained!")
print()
print("=== WHAT THE MODEL LEARNED (coefficients) ===")
for feat, coef in zip(FEATURES, model_lr.coef_[0]):
    direction = "MORE anomalous" if coef > 0 else "MORE benign"
    print(f"  {feat:22s}: {coef:+.4f}  →  higher value = {direction}")
print()
print("KEY INSIGHT:")
print("  keyword_count is the strongest predictor (+0.70)")
print("  payload_length is NEGATIVE — longer = more likely benign")
print("  (because XSS benign rows are long Wikipedia HTML)")
print("  This is CORRECT behaviour based on data")

LogisticRegression trained!

=== WHAT THE MODEL LEARNED (coefficients) ===
  payload_length        : -0.0263  →  higher value = MORE benign
  special_char_count    : +0.2777  →  higher value = MORE anomalous
  keyword_count         : +0.7026  →  higher value = MORE anomalous

KEY INSIGHT:
  keyword_count is the strongest predictor (+0.70)
  payload_length is NEGATIVE — longer = more likely benign
  (because XSS benign rows are long Wikipedia HTML)
  This is CORRECT behaviour based on data


In [6]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model_lr.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("=" * 55)
print("LOGISTIC REGRESSION RESULTS — REAL DATA")
print("=" * 55)
print(f"Accuracy : {accuracy * 100:.1f}%")
print()
print(classification_report(y_test, y_pred, target_names=['Benign','Attack']))
print()
cm = confusion_matrix(y_test, y_pred)
print("=== CONFUSION MATRIX ===")
print(f"                  Pred Benign  Pred Attack")
print(f"True Benign       {cm[0][0]:>10,}  {cm[0][1]:>11,}")
print(f"True Attack       {cm[1][0]:>10,}  {cm[1][1]:>11,}")
print()
print("WHY 75% ACCURACY (not higher):")
print("  XSS benign rows = Wikipedia HTML with HTML tags")
print("  Some HTML tags look like XSS patterns to the model")
print("  Better XSS dataset = 90%+ accuracy")

LOGISTIC REGRESSION RESULTS — YOUR REAL DATA
Accuracy : 75.0%

              precision    recall  f1-score   support

      Benign       0.76      0.80      0.78      4624
      Attack       0.73      0.69      0.71      3740

    accuracy                           0.75      8364
   macro avg       0.75      0.74      0.75      8364
weighted avg       0.75      0.75      0.75      8364


=== CONFUSION MATRIX ===
                  Pred Benign  Pred Attack
True Benign            3,685          939
True Attack            1,153        2,587

WHY 75% ACCURACY (not higher):
  XSS benign rows = Wikipedia HTML with HTML tags
  Some HTML tags look like XSS patterns to the model
  Better XSS dataset = 90%+ accuracy


In [7]:
print("=== TEST ON PAYLOADS FROM ACTUAL FILES ===")
test_cases = [
    ("Normal input",        "login=admin&password=secret123",                    0),
    ("SQLi OR bypass",      "' OR 1=1--",                                        1),
    ("SQLi UNION SELECT",   "' UNION SELECT username,password FROM users--",     1),
    ("SQLi from your file", '" or pg_sleep  (  __TIME__  )  --',                 1),
    ("XSS onmouseover",     'test',              1),
    ("XSS onerror",         "",        1),
    ("XSS from your file",  "XSS",     1),
    ("Normal search",       "search=laptop&brand=dell&price=500",                0),
]

print(f"  {'Type':25s}  {'True':8s}  {'Pred':8s}  {'Prob':5s}  {'OK?'}")
print("-" * 68)
for name, payload, true_label in test_cases:
    pl, sc, kw = extract_features(payload)
    X_t  = pd.DataFrame([[pl, sc, kw]], columns=FEATURES)
    pred = int(model_lr.predict(X_t)[0])
    prob = model_lr.predict_proba(X_t)[0][1]
    t_s  = "ATTACK" if true_label==1 else "benign"
    p_s  = "ATTACK" if pred==1 else "benign"
    res  = "CORRECT" if pred==true_label else "wrong"
    print(f"  {name:25s}  {t_s:8s}  {p_s:8s}  {prob:.2f}  {res}")

=== TEST ON PAYLOADS FROM ACTUAL FILES ===
  Type                       True      Pred      Prob   OK?
--------------------------------------------------------------------
  Normal input               benign    benign    0.16  CORRECT
  SQLi OR bypass             ATTACK    ATTACK    0.53  CORRECT
  SQLi UNION SELECT          ATTACK    ATTACK    0.52  CORRECT
  SQLi from your file        ATTACK    benign    0.45  wrong
  XSS onmouseover            ATTACK    benign    0.18  wrong
  XSS onerror                ATTACK    benign    0.20  wrong
  XSS from your file         ATTACK    benign    0.18  wrong
  Normal search              benign    benign    0.19  CORRECT
